# Logistic Regression from Scratch
# Student Pass / Fail Prediction

---

**Repository:** Machine Learning from Scratch  
**Notebook:** 01 — Logistic Regression  
**Algorithm:** Logistic Regression via Gradient Descent (no Scikit-Learn)  
**Target:** Binary — Pass (1) if final_score >= 70, Fail (0) otherwise  
**Author:** Ather-ops  

---

## Objective

Predict whether a student **passes or fails** their exam based on three behavioral features: study hours, sleep hours, and attendance. We implement the full logistic regression pipeline from scratch — no Scikit-Learn anywhere.

---

## Why Logistic Regression, Not Linear Regression?

Linear regression predicts a continuous number like 74.5 or 38.2. But here we want a binary decision — Pass or Fail. If we used linear regression, it could predict values like -5 or 120, which are meaningless for a probability.

Logistic regression solves this by passing the linear output through the **sigmoid function**, which squashes any number into the range (0, 1) — a valid probability.

---

## The Full Math Pipeline

### Step 1 — Linear Combination

$$z = m_1 \cdot study\_hours + m_2 \cdot sleep\_hours + m_3 \cdot attendance + b$$

### Step 2 — Sigmoid Activation

$$\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

$\hat{p}$ is the predicted probability of passing.

### Step 3 — Binary Cross-Entropy Loss

$$L = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

### Step 4 — Gradients

$$\frac{\partial L}{\partial m_1} = \frac{1}{n} \sum (\hat{p} - y) \cdot x_1$$

$$\frac{\partial L}{\partial b} = \frac{1}{n} \sum (\hat{p} - y)$$

### Step 5 — Parameter Update

$$m_1 \leftarrow m_1 - \alpha \cdot \frac{\partial L}{\partial m_1}$$

$$b \leftarrow b - \alpha \cdot \frac{\partial L}{\partial b}$$

---

## Full Pipeline

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Load dataset |
| 3 | Initial visualization — raw scatter plots |
| 4 | EDA — statistics and missing values |
| 5 | Outlier detection — IQR method |
| 6 | Create binary target — Pass / Fail |
| 7 | Feature and target selection + quantile bucketing |
| 8 | Train-test split (manual) |
| 9 | Manual standardization |
| 10 | Define sigmoid, log loss, standardize functions |
| 11 | Initialize parameters |
| 12 | Gradient descent training loop |
| 13 | Loss curve visualization |
| 14 | Predictions and probabilities |
| 15 | Confusion matrix from scratch |
| 16 | Evaluation metrics |
| 17 | Post-training visualization |
| 18 | Predict for a new student |

---
## Step 1 — Import Libraries

Only three libraries. No Scikit-Learn, no `LogisticRegression`, no `accuracy_score`, no `confusion_matrix`. Every operation is written by hand.

| Library | Role |
|---------|------|
| `pandas` | Load CSV, inspect data, quantile bucketing |
| `numpy` | Sigmoid, log loss, gradients, array operations |
| `matplotlib.pyplot` | All scatter plots, loss curve, probability bars |

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

---
## Step 2 — Load Dataset

We load the student scores CSV and immediately inspect the first 10 rows.

**Expected columns:**

| Column | Type | Description |
|--------|------|-------------|
| `study_hours` | float | Daily hours spent studying |
| `sleep_hours` | float | Daily hours of sleep |
| `attendance` | float | Class attendance percentage |
| `final_score` | float | Exam score — will be converted to binary in Step 6 |

In [ ]:
# step 1: Load Data

df=pd.read_csv("student_scores_csv.txt")
print("original data :")
print("="*60)
print(df.head(10))
print("="*60)

---
## Step 3 — Initial Visualization

Before any transformation, we plot each feature against `final_score` to understand the raw relationships.

**What to look for:**

| Plot | Expected pattern |
|------|------------------|
| Study Hours vs Final Score | Upward trend — more study, higher score |
| Sleep Hours vs Final Score | Mild upward — rest helps performance |
| Attendance vs Final Score | Upward — more class time, better outcomes |

Note: we are plotting against the raw `final_score` here. After Step 6, the target becomes binary (Pass=1 / Fail=0) and the scatter plots will look different.

In [ ]:
# step 2: Initail visualisation
plt.figure(figsize=(15,4))
# 1:STUDY HOURS VS FINAL SCORE
plt.subplot(1,3,1)
plt.scatter(df["study_hours"],df["final_score"],color="red")
plt.xlabel("Study Hours")
plt.ylabel("Final Score")
plt.title("Study Hours VS Final Score(Before Training)")
# 2: SLEPP HOURS VS FINAL SCORE
plt.subplot(1,3,2)
plt.scatter(df["sleep_hours"],df["final_score"],color="yellow")
plt.xlabel("Sleep Hours")
plt.ylabel("Final score")
plt.title("Sleep hours VS Final score(Before Training)")
# 3: ATTENDANCE VS FINAL SCORE 
plt.subplot(1,3,3)
plt.scatter(df["attendance"],df["final_score"],color="green")
plt.xlabel("Attendance")
plt.ylabel("Final Score")
plt.title("Attendance VS Final score (Before Training)")
plt.tight_layout()
plt.show()

---
## Step 4 — EDA — Descriptive Statistics and Missing Values

### `df.describe()` — Statistical Summary

| Statistic | What to check |
|-----------|---------------|
| `count` | If any column has fewer rows than others, it has missing values |
| `mean` | Average study hours of 8-12 is plausible; 0.5 would be suspicious |
| `min` | Negative study hours or scores below 0 are data errors |
| `max` | Scores above 100 or attendance above 100 are impossible |
| `std` | Very high std relative to mean suggests outliers |

### `df.isnull().sum()` — Missing Values

Any non-zero value must be handled before training. Logistic regression cannot process `NaN` values.

In [ ]:
# step 3: Data expoloration(EDA)
print("Basic statistic:",df.describe())
print("="*80)
print("missing values:",df.isnull().sum)
print("="*80)

---
## Step 5 — Outlier Detection — IQR Method

We apply the Interquartile Range method to detect outliers in `study_hours`.

**Formula:**

$$IQR = Q3 - Q1$$

$$\text{Lower Bound} = Q1 - 1.5 \times IQR$$

$$\text{Upper Bound} = Q3 + 1.5 \times IQR$$

Any student whose study hours fall outside these fences is printed as an outlier.

**Why check study hours specifically:**  
Study hours is the most influential feature for predicting pass/fail. An outlier here — a student logged with 23 hours per day — would distort the learned weights more than an outlier in sleep or attendance.

In [ ]:
# step 4: Finding outliers
Q1=df["study_hours"].quantile(0.25)
Q3=df["study_hours"].quantile(0.75)

IQR=Q3-Q1

lower_bound=Q1-1.5*IQR
upper_bound=Q3+1.5*IQR

outliers=df[(df["study_hours"]<lower_bound)|(df["study_hours"]>upper_bound)]
print("outlier:",outliers)

print("="*80)
print(f"Q1           : {Q1}")
print(f"Q3           : {Q3}")
print(f"IQR          : {IQR}")
print(f"Lower Bound  : {lower_bound}")
print(f"Upper Bound  : {upper_bound}")
print(f"Outliers found: {len(outliers)}")
print("="*80)

---
## Step 6 — Create Binary Target

Logistic regression requires a binary target — not a continuous score. We apply a threshold:

$$Y = \begin{cases} 1 & \text{if } final\_score \geq 70 \quad (\text{Pass}) \\ 0 & \text{if } final\_score < 70 \quad (\text{Fail}) \end{cases}$$

**Why 70?**  
70 is a standard academic pass threshold. Students scoring 70 or above are classified as Pass (1), below as Fail (0). This converts our regression problem into a proper binary classification problem that logistic regression is designed for.

**This is the key difference from the linear regression notebooks:**  
In Notebooks 01-03 we predicted the actual score. Here we predict whether the student passes or fails. The model output is a probability, not a number.

In [ ]:
# Create binary target — Pass = 1, Fail = 0
df["pass_fail"] = (df["final_score"] >= 70).astype(int)

print("Binary Target Created")
print("="*60)
print(f"Threshold     : final_score >= 70 = Pass (1)")
print(f"Total students: {len(df)}")
print(f"Pass  (1)     : {df['pass_fail'].sum()}")
print(f"Fail  (0)     : {(df['pass_fail'] == 0).sum()}")
print(f"Pass rate     : {df['pass_fail'].mean()*100:.1f}%")
print("="*60)
print(df[["study_hours","sleep_hours","attendance","final_score","pass_fail"]].head(10))
print("="*60)

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution bar
counts = df["pass_fail"].value_counts().sort_index()
axes[0].bar(["Fail (0)", "Pass (1)"], counts.values,
            color=["tomato", "steelblue"], edgecolor="white", width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.5, str(v), ha="center", fontsize=12)
axes[0].set_title("Pass / Fail Class Distribution")
axes[0].set_ylabel("Count")
axes[0].grid(True, axis="y", linestyle="--", alpha=0.5)

# Study hours colored by pass/fail
pass_df = df[df["pass_fail"] == 1]
fail_df = df[df["pass_fail"] == 0]
axes[1].scatter(fail_df["study_hours"], fail_df["final_score"],
                color="tomato", label="Fail", s=70, edgecolors="white")
axes[1].scatter(pass_df["study_hours"], pass_df["final_score"],
                color="steelblue", label="Pass", s=70, edgecolors="white")
axes[1].axhline(70, color="black", linestyle="--",
                linewidth=1.5, label="Threshold = 70")
axes[1].set_xlabel("Study Hours")
axes[1].set_ylabel("Final Score")
axes[1].set_title("Pass/Fail Decision Boundary (score=70)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

---
## Step 7 — Feature Selection and Quantile Bucketing

We select the three features and the new binary target. We also add quantile bucketing on study hours — this adds an interpretable grouping column for analysis.

**`pd.qcut()` vs `pd.cut()`:**

| Method | Boundary logic | Result |
|--------|---------------|--------|
| `pd.cut()` | Fixed value boundaries | Bins may have very different sizes |
| `pd.qcut()` | Equal-frequency boundaries | Each bin has roughly the same number of students |

With `q=3`, students are divided into three equal-sized groups: bottom third (low study hours), middle third, top third (high study hours). This is for analysis only — not used as a model feature.

In [ ]:
# step 5: TARGET AND FEATURE
X=df[["study_hours","sleep_hours","attendance"]]
Y=df["pass_fail"]   # binary target: 1=Pass, 0=Fail
# Quantile buckting
df["student_quantile"]=pd.qcut(df["study_hours"],q=3,labels=["Q1_small","Q2_medium","Q3_large"])
print("Data after quantile bucting:")
print("="*80)
print(df[["study_hours","sleep_hours","attendance","final_score","pass_fail","student_quantile"]].head(10))
print("="*80)

print("Pass rate per study quantile:")
print(df.groupby("student_quantile")["pass_fail"].mean().round(3))
print("="*80)

---
## Step 8 — Manual Train-Test Split

We split the data manually using NumPy random indexing — no `train_test_split` from Scikit-Learn.

**How manual split works:**
1. Generate a shuffled array of indices `[0, 1, 2, ..., n-1]`
2. Take the first 80% as training indices, the remaining 20% as test indices
3. Use `.iloc[]` to select the corresponding rows

**Why shuffle first:**  
If the data is ordered (e.g. all low-scorers at the top), a simple slice without shuffling would create a biased test set. Shuffling ensures both sets are representative.

`np.random.seed(42)` ensures the same split every time the notebook runs — reproducibility is essential for comparing results.

In [ ]:
#step 6: TRAIN TEST SPLIT — manual, no sklearn
np.random.seed(42)
n          = len(df)
indices    = np.random.permutation(n)
split      = int(0.8 * n)
train_idx  = indices[:split]
test_idx   = indices[split:]

X_train = X.iloc[train_idx].values
X_test  = X.iloc[test_idx].values
Y_train = Y.iloc[train_idx].values
Y_test  = Y.iloc[test_idx].values

print("Train-Test Split")
print("="*60)
print(f"Total students : {n}")
print(f"Training set   : {len(X_train)} students")
print(f"Test set       : {len(X_test)} students")
print("="*60)
print(f"Training Pass rate : {Y_train.mean()*100:.1f}%")
print(f"Test     Pass rate : {Y_test.mean()*100:.1f}%")
print("="*60)

---
## Step 9 — Manual Feature Standardization

We standardize the features using the same formula as `StandardScaler` — but written by hand:

$$x_{scaled} = \frac{x - \mu_{train}}{\sigma_{train}}$$

**Critical rule — fit only on training data:**

| Step | Data | Operation |
|------|------|----------|
| Compute mean and std | Training set only | `mean_train`, `std_train` |
| Scale training data | Training set | `(X_train - mean_train) / std_train` |
| Scale test data | Test set | `(X_test - mean_train) / std_train` |

We use the **training mean and std on the test data** — never compute new statistics from the test set. Computing test-set statistics would leak information about unseen data into the normalization step.

In [ ]:
# step 7: Feature scaling — manual standardization (no StandardScaler)
mean_train = X_train.mean(axis=0)
std_train  = X_train.std(axis=0)

X_train_scaled = (X_train - mean_train) / std_train
X_test_scaled  = (X_test  - mean_train) / std_train

print("Manual Standardization")
print("="*60)
print(f"Feature order : study_hours, sleep_hours, attendance")
print(f"Training mean : {mean_train.round(4)}")
print(f"Training std  : {std_train.round(4)}")
print("="*60)
print("X_train_scaled — first 5 rows:")
print(X_train_scaled[:5].round(4))
print("="*60)
print(f"Post-scaling mean (should be ~0): {X_train_scaled.mean(axis=0).round(6)}")
print(f"Post-scaling std  (should be ~1): {X_train_scaled.std(axis=0).round(6)}")
print("="*60)

---
## Step 10 — Define Helper Functions

We define the three mathematical building blocks of logistic regression as standalone functions. Writing them as named functions rather than inline expressions makes the training loop clean and auditable.

### `sigmoid(z)`

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Converts any real number into a probability between 0 and 1.

### `log_loss(y_true, y_pred)`

$$L = -\frac{1}{n} \sum \left[ y \log(\hat{p}) + (1-y) \log(1-\hat{p}) \right]$$

`np.clip` prevents `log(0)` from producing `-inf` and crashing the computation.

### Gradient formulas

The partial derivatives of log loss with respect to each weight have a beautiful closed form — the sigmoid and log derivatives cancel:

$$\nabla m_j = \frac{1}{n} \sum (\hat{p}_i - y_i) \cdot x_{ji}$$

$$\nabla b = \frac{1}{n} \sum (\hat{p}_i - y_i)$$

The error term is simply `(predicted_probability - actual_label)` — identical in form to the linear regression gradient.

In [ ]:
# step 8: Define mathematical functions

def sigmoid(z):
    """Squash any value to (0, 1) — the probability output."""
    return 1 / (1 + np.exp(-z))

def log_loss(y_true, y_pred):
    """Binary cross-entropy loss. Clip prevents log(0) = -inf."""
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Verify sigmoid
print("Sigmoid verification:")
print("="*60)
for z_val in [-5, -2, 0, 2, 5]:
    print(f"  sigmoid({z_val:+3d}) = {sigmoid(z_val):.6f}")
print("  Approaches 0 for large negative z, 0.5 at z=0, 1.0 for large positive z")
print("="*60)

---
## Step 11 — Initialize Parameters

All weights start at zero. With all parameters at zero, $z = 0$ for every student, so `sigmoid(0) = 0.5` — the model begins completely uncertain about every prediction.

| Parameter | Initial | What it learns |
|-----------|---------|----------------|
| `m1` | 0.0 | How study hours influence pass probability |
| `m2` | 0.0 | How sleep hours influence pass probability |
| `m3` | 0.0 | How attendance influences pass probability |
| `b` | 0.0 | Base log-odds when all features are at their mean |

**Learning rate 0.1, 2000 epochs:**  
After standardization, features are on the same scale, so a learning rate of 0.1 is safe. 2000 epochs gives the log loss enough iterations to converge — it is slower than MSE because the sigmoid gradient flattens near 0 and 1.

In [ ]:
# step 8: Model selection — parameters initialized
m1 = m2 = m3 = b = 0.0
lr           = 0.1
epochs       = 2000
n_train      = len(Y_train)
loss_history = []

# Extract feature columns
X1_train = X_train_scaled[:, 0]   # study_hours
X2_train = X_train_scaled[:, 1]   # sleep_hours
X3_train = X_train_scaled[:, 2]   # attendance

X1_test  = X_test_scaled[:, 0]
X2_test  = X_test_scaled[:, 1]
X3_test  = X_test_scaled[:, 2]

print("Parameters Initialized")
print("="*60)
print(f"m1 = m2 = m3 = b = 0.0")
print(f"Learning rate  : {lr}")
print(f"Epochs         : {epochs}")
print(f"Training size  : {n_train}")
print("="*60)
# Initial prediction before any training
z_init = m1*X1_train + m2*X2_train + m3*X3_train + b
p_init = sigmoid(z_init)
print(f"Initial prediction (all students) : {p_init[0]:.4f}  (all equal 0.5 — no knowledge yet)")
print(f"Initial log loss                  : {log_loss(Y_train, p_init):.4f}")
print("="*60)

---
## Step 12 — Gradient Descent Training Loop

The full training loop — 2000 epochs of gradient descent on the training set.

**Every epoch does exactly 7 things:**

```
1. z      = m1*X1 + m2*X2 + m3*X3 + b       (linear combination)
2. y_pred = sigmoid(z)                        (probability output)
3. loss   = log_loss(Y_train, y_pred)         (how wrong are we)
4. dm1    = (1/n) * sum(X1 * (y_pred - Y))   (gradient for m1)
5. dm2    = (1/n) * sum(X2 * (y_pred - Y))   (gradient for m2)
6. dm3    = (1/n) * sum(X3 * (y_pred - Y))   (gradient for m3)
7. db     = (1/n) * sum(y_pred - Y)           (gradient for b)
   then:  m1 -= lr * dm1  etc.
```

Print every 200 epochs to monitor without flooding the output.

In [ ]:
# step 8: Training loop — gradient descent
for epoch in range(epochs):
    # Forward pass
    z      = m1*X1_train + m2*X2_train + m3*X3_train + b
    y_pred = sigmoid(z)

    # Loss
    loss = log_loss(Y_train, y_pred)
    loss_history.append(loss)

    # Gradients
    error = y_pred - Y_train
    dm1   = (1/n_train) * np.sum(X1_train * error)
    dm2   = (1/n_train) * np.sum(X2_train * error)
    dm3   = (1/n_train) * np.sum(X3_train * error)
    db    = (1/n_train) * np.sum(error)

    # Update parameters
    m1 -= lr * dm1
    m2 -= lr * dm2
    m3 -= lr * dm3
    b  -= lr * db

    if epoch % 200 == 0:
        print(f"Epoch {epoch:>4} | Loss: {loss:.4f} | m1:{m1:.3f} m2:{m2:.3f} m3:{m3:.3f} b:{b:.3f}")

---
## Step 13 — Loss Curve

We plot the log loss across all 2000 training epochs and print the final learned weights.

**Expected pattern:**
- Loss starts around `ln(2) ≈ 0.693` — this is the log loss when all predictions are 0.5 (maximum uncertainty)
- Loss decreases as the model learns to separate Pass from Fail students
- Curve flattens as parameters approach their optimal values

**Reading the final weights:**  
Because features are standardized, the weight magnitudes are directly comparable. The feature with the largest absolute weight has the strongest influence on predicting Pass or Fail.

In [ ]:
# Loss curve
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(loss_history, color="steelblue", linewidth=1.5)
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Log Loss")
axes[0].set_title("Training Loss Curve — All 2000 Epochs")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].plot(loss_history[:400], color="tomato", linewidth=1.5)
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Log Loss")
axes[1].set_title("First 400 Epochs — Initial Drop")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print("="*60)
print(f"Initial loss    : {loss_history[0]:.4f}")
print(f"Loss at ep 200  : {loss_history[200]:.4f}")
print(f"Loss at ep 1000 : {loss_history[1000]:.4f}")
print(f"Final loss      : {loss_history[-1]:.4f}")
print(f"Loss reduction  : {(1-loss_history[-1]/loss_history[0])*100:.2f}%")
print("="*60)
print("Final Learned Parameters:")
print(f"  m1 (study_hours)  : {m1:.4f}")
print(f"  m2 (sleep_hours)  : {m2:.4f}")
print(f"  m3 (attendance)   : {m3:.4f}")
print(f"  b  (bias)         : {b:.4f}")
print("="*60)
ranked = sorted([("study_hours",abs(m1)),("sleep_hours",abs(m2)),("attendance",abs(m3))],
                key=lambda x: x[1], reverse=True)
print("Feature influence ranking:")
for r,(feat,w) in enumerate(ranked,1):
    print(f"  {r}. {feat:15s}: |weight| = {w:.4f}")
print("="*60)

---
## Step 14 — Predictions and Probabilities

We generate predictions on the **test set** — students the model never saw during training.

**Two outputs:**

| Output | Method | Value |
|--------|--------|-------|
| `y_prob` | `sigmoid(z)` | Probability of passing (0 to 1) |
| `y_pred` | `y_prob >= 0.5` | Binary decision — Pass (1) or Fail (0) |

Threshold = 0.5 means: if the model is at least 50% confident the student passes, predict Pass. This is the default for balanced classes.

In [ ]:
# step 9: Predictions
z_test  = m1*X1_test + m2*X2_test + m3*X3_test + b
y_prob  = sigmoid(z_test)
y_pred  = (y_prob >= 0.5).astype(int)

# Prediction table
result_df = pd.DataFrame({
    "study_hours" : X_test[:, 0].round(1),
    "sleep_hours" : X_test[:, 1].round(1),
    "attendance"  : X_test[:, 2].round(1),
    "Actual"      : Y_test,
    "P(Pass)"     : y_prob.round(4),
    "Predicted"   : y_pred,
    "Correct"     : (Y_test == y_pred).astype(int)
})

print("Test Set Predictions")
print("="*80)
print(result_df.to_string(index=False))
print("="*80)
print(f"Correct : {result_df['Correct'].sum()} / {len(result_df)}")
print("="*80)

---
## Step 15 — Confusion Matrix from Scratch

We count TP, TN, FP, FN manually — the same way as Notebook 04.

```
Predicted Fail    Predicted Pass
Actual Fail  |    TN    |    FP    |
Actual Pass  |    FN    |    TP    |
```

**In student pass/fail prediction:**
- **FN (False Negative)** — predicted Fail, actual Pass — the model discouraged a student who would have passed
- **FP (False Positive)** — predicted Pass, actual Fail — the model gave false confidence to a student who failed

Both errors have academic consequences. FN may cause the student to give up; FP may cause them not to seek help in time.

In [ ]:
# stepn 10: Confusion Matrix — manual
TP = TN = FP = FN = 0
for actual, pred in zip(Y_test, y_pred):
    if   actual == 1 and pred == 1: TP += 1
    elif actual == 0 and pred == 0: TN += 1
    elif actual == 0 and pred == 1: FP += 1
    elif actual == 1 and pred == 0: FN += 1

print("Confusion Matrix:")
print(f"TN:{TN} FP:{FP}")
print(f"FN:{FN} TP:{TP}")
print("="*60)
print(f"True  Positives (TP) : {TP}  — predicted Pass, actual Pass")
print(f"True  Negatives (TN) : {TN}  — predicted Fail, actual Fail")
print(f"False Positives (FP) : {FP}  — predicted Pass, actual Fail")
print(f"False Negatives (FN) : {FN}  — predicted Fail, actual Pass")
print("="*60)

---
## Step 16 — Evaluation Metrics

All metrics computed manually from TP, TN, FP, FN:

| Metric | Formula | Meaning |
|--------|---------|--------|
| Accuracy | $\frac{TP+TN}{Total}$ | Overall fraction correctly classified |
| Precision | $\frac{TP}{TP+FP}$ | Of all predicted Pass, what fraction actually passed? |
| Recall | $\frac{TP}{TP+FN}$ | Of all actual Pass students, what fraction did we catch? |
| F1 Score | $\frac{2 \times P \times R}{P+R}$ | Harmonic mean of Precision and Recall |

**Why accuracy alone is misleading:**  
If 90% of students pass, a model that always predicts Pass achieves 90% accuracy without learning anything. F1 score captures the quality of both Pass and Fail predictions.

In [ ]:
# stepn 10: Evalution — all from scratch
accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = (2*precision*recall)/(precision+recall) if (precision+recall) > 0 else 0

print("accuracy:",accuracy)
print("Confusion matrix:")
print(f"[[TN={TN} FP={FP}]")
print(f" [FN={FN} TP={TP}]]")
print("="*80)
print(f"Accuracy  : {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"Precision : {precision:.4f}  ({precision*100:.1f}% of predicted Pass were real Pass)")
print(f"Recall    : {recall:.4f}  ({recall*100:.1f}% of actual Pass students caught)")
print(f"F1 Score  : {f1:.4f}")
print("="*80)

---
## Step 17 — Post-Training Visualization

We plot each feature against the binary target for the test set — overlaying actual labels and model predictions.

**What changes from the before-training plots:**
- Before: y-axis showed raw `final_score` (continuous, 0-100)
- After: y-axis shows binary `pass_fail` label (0 or 1)
- The pink dots are actual labels (0 or 1), the green dots are predicted labels (0 or 1)

Where pink and green overlap — correct prediction. Where they separate — error.

In [ ]:
# step 11: Visuallisation after predictions
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
# 1: STUDY HOURS VS PASS/FAIL
plt.scatter(X_test[:,0],Y_test,label="Actual",color="pink")
plt.scatter(X_test[:,0],y_pred,label="Predicted",color="green",alpha=0.6)
plt.title("STUDY HOURS VS PASS/FAIL (After Training)")
plt.xlabel("STUDY HOURS (scaled)")
plt.ylabel("PASS (1) / FAIL (0)")
plt.grid(True)
plt.legend()
# 2: SLEEP HOURS VS PASS/FAIL
plt.subplot(1,3,2)
plt.scatter(X_test[:,1],Y_test,color="black",label="Actual")
plt.scatter(X_test[:,1],y_pred,label="Predicted",color="blue",alpha=0.6)
plt.xlabel("SLEEP HOURS (scaled)")
plt.ylabel("PASS (1) / FAIL (0)")
plt.title("SLEEP HOURS VS PASS/FAIL (After Training)")
plt.grid(True)
plt.legend()
# 3: ATTENDANCE VS PASS/FAIL
plt.subplot(1,3,3)
plt.scatter(X_test[:,2],Y_test,color="red",label="Actual")
plt.scatter(X_test[:,2],y_pred,label="Predicted",color="orange",alpha=0.6)
plt.xlabel("ATTENDANCE (scaled)")
plt.ylabel("PASS (1) / FAIL (0)")
plt.title("ATTENDANCE VS PASS/FAIL (After Training)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
print("="*100)

---
## Step 17b — Confusion Matrix Heatmap and Probability Bars

Two additional diagnostic plots that are standard for any classification problem:

1. **Confusion matrix heatmap** — colored grid showing TN, FP, FN, TP counts
2. **Probability bars** — predicted probability of passing for each test student, colored by actual label

The probability bar is the most information-rich single plot — it shows not just whether the model was right, but how confident it was.

In [ ]:
# Confusion matrix heatmap and probability bars
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix heatmap
cm_array = np.array([[TN, FP], [FN, TP]])
im = axes[0].imshow(cm_array, cmap="Blues")
plt.colorbar(im, ax=axes[0])
cell_labels = [["TN", "FP"], ["FN", "TP"]]
for i in range(2):
    for j in range(2):
        axes[0].text(j, i,
                     f"{cell_labels[i][j]}\n{cm_array[i,j]}",
                     ha="center", va="center", fontsize=13, fontweight="bold",
                     color="white" if cm_array[i,j] > cm_array.max()/2 else "black")
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Predicted Fail", "Predicted Pass"], fontsize=10)
axes[0].set_yticklabels(["Actual Fail", "Actual Pass"], fontsize=10)
axes[0].set_title("Confusion Matrix", fontsize=12)

# Probability bars
bar_colors = ["steelblue" if y == 1 else "tomato" for y in Y_test]
bars = axes[1].bar(range(len(y_prob)), y_prob,
                   color=bar_colors, edgecolor="white", width=0.6)
axes[1].axhline(0.5, color="black", linestyle="--",
                linewidth=1.5, label="Decision boundary (0.5)")
for i, (bar, prob) in enumerate(zip(bars, y_prob)):
    axes[1].text(i, prob + 0.02, f"{prob:.2f}",
                 ha="center", fontsize=8)
axes[1].set_xticks(range(len(y_prob)))
axes[1].set_xticklabels([f"S{i+1}" for i in range(len(y_prob))], fontsize=9)
axes[1].set_xlabel("Test Student")
axes[1].set_ylabel("P(Pass)")
axes[1].set_ylim(0, 1.2)
axes[1].set_title("Predicted Pass Probabilities\n(Blue=Actual Pass, Red=Actual Fail)")
axes[1].legend(fontsize=9)
axes[1].grid(True, axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("="*60)

---
## Step 18 — Predict for a New Student

We use the final learned parameters to predict the pass/fail outcome for a new student.

**New student profile:**

| Feature | Value | Context |
|---------|-------|---------|
| study_hours | 7 | Above average |
| sleep_hours | 8 | Good |
| attendance | 85% | Decent |

**Pipeline for new input:**
1. Apply the **training mean and std** to standardize — never recompute from the new input
2. Compute $z = m_1 x_1 + m_2 x_2 + m_3 x_3 + b$
3. Apply `sigmoid(z)` to get pass probability
4. Apply threshold 0.5 to get the final binary decision

In [ ]:
# step 12: Predicting new values
new_study      = 7
new_sleep      = 8
new_attendance = 85

# Standardize using TRAINING mean and std
new_x1 = (new_study      - mean_train[0]) / std_train[0]
new_x2 = (new_sleep      - mean_train[1]) / std_train[1]
new_x3 = (new_attendance - mean_train[2]) / std_train[2]

new_z    = m1*new_x1 + m2*new_x2 + m3*new_x3 + b
new_prob = sigmoid(new_z)
new_pred = int(new_prob >= 0.5)
label    = "PASS" if new_pred == 1 else "FAIL"

print("New Student Prediction")
print("="*60)
print(f"Study Hours  : {new_study}")
print(f"Sleep Hours  : {new_sleep}")
print(f"Attendance   : {new_attendance}%")
print("="*60)
print(f"Standardized : [{new_x1:.4f}, {new_x2:.4f}, {new_x3:.4f}]")
print(f"Linear z     : {new_z:.4f}")
print(f"P(Pass)      : {new_prob:.4f}  ({new_prob*100:.1f}%)")
print(f"Decision     : {label}")
print("="*60)

---
## Pipeline Summary

| Step | Action | Key concept |
|------|--------|-------------|
| 1 | Import pandas, numpy, matplotlib | No Scikit-Learn |
| 2 | Load student_scores_csv.txt | Four columns including final_score |
| 3 | Scatter plots — raw continuous data | Before/after comparison baseline |
| 4 | describe() and isnull() | Data quality check |
| 5 | IQR outlier detection on study_hours | Q1, Q3, fences, outlier rows printed |
| 6 | Convert final_score to binary — score >= 70 = Pass | Regression problem becomes classification |
| 7 | Feature selection + pd.qcut quantile bucketing | X = 3 features, Y = pass_fail |
| 8 | Manual train-test split — np.random.permutation | 80/20, seed=42, no sklearn |
| 9 | Manual standardization — training mean/std only | No StandardScaler |
| 10 | Define sigmoid and log_loss functions | Mathematical building blocks |
| 11 | Initialize m1=m2=m3=b=0, lr=0.1 | All start at 0.5 probability |
| 12 | 2000-epoch gradient descent | Forward pass, loss, gradients, update |
| 13 | Loss curve — full + early zoom | Convergence confirmed visually |
| 14 | y_prob = sigmoid(z), y_pred = threshold | Full prediction table |
| 15 | Confusion matrix — manual loop | TP, TN, FP, FN by hand |
| 16 | Accuracy, Precision, Recall, F1 | All from scratch |
| 17 | Post-training scatter plots + heatmap + probability bars | Three visualization types |
| 18 | New student prediction | Standardize with training stats |

---

## What Scikit-Learn Was Doing

The original notebook used:
```python
model = LogisticRegression()
model.fit(X_train_scaled, Y_train)
y_pred = model.predict(X_test_scaled)
```

This notebook shows what those three lines do internally:
- `model.fit` = 2000 iterations of the gradient loop above
- `model.predict` = sigmoid(z) followed by threshold comparison
- `accuracy_score` = `(TP + TN) / Total` computed over the test set
- `confusion_matrix` = the four-branch loop counting TP, TN, FP, FN

---

**Next notebook:** `02_logistic_regression_multiclass_scratch.ipynb` — Extending to more than two classes using Softmax